# Examine drop-out risks and protective factors
We will use a combination of 
1. XGBoost global feature importance
2. Permutation importance
3. SHAP global summary

and compare the results to derive insights on drop-out risks and protective factors. We combine three complementary methods to ensure robustness and validity of our findings.

Note that none of these methods provide causal insights, but they can help *identify important factors* associated with student drop-out.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path

print("Libraries imported successfully!")

In [ ]:
# Load the tuned model
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
final_model = pd.read_pickle(PROCESSED_DIR / "xgb_final_tuned_trainval.pkl")

print(f"Model loaded successfully!")

## 1. XGBoost Global Feature Importance

Global feature importance shows which features are most important for the model's decisions across all predictions. We'll use the 'gain' metric, which measures the average improvement in model performance when a feature is used for splitting.

In [ ]:
# Extract XGBoost global feature importance
# Get importance scores for different metrics
importance_gain = final_model.get_booster().get_score(importance_type='gain')
importance_weight = final_model.get_booster().get_score(importance_type='weight')
importance_cover = final_model.get_booster().get_score(importance_type='cover')

# Convert to DataFrame for easier analysis
def importance_to_df(importance_dict, importance_type):
    df = pd.DataFrame(list(importance_dict.items()), 
                     columns=['feature', f'importance_{importance_type}'])
    df = df.sort_values(f'importance_{importance_type}', ascending=False)
    return df

df_gain = importance_to_df(importance_gain, 'gain')
df_weight = importance_to_df(importance_weight, 'weight')
df_cover = importance_to_df(importance_cover, 'cover')

In [ ]:
# Visualize XGBoost global feature importance
plt.figure(figsize=(12, 12))

# Plot top 20 features by gain
top_20_gain = df_gain.head(20)
plt.subplot(3, 1, 1)
plt.barh(range(len(top_20_gain)), top_20_gain['importance_gain'])
plt.yticks(range(len(top_20_gain)), top_20_gain['feature'])
plt.xlabel('Importance Score (Gain)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Gain)')
plt.gca().invert_yaxis()

# Plot top 20 features by weight  
top_20_weight = df_weight.head(20)
plt.subplot(3, 1, 2)
plt.barh(range(len(top_20_weight)), top_20_weight['importance_weight'])
plt.yticks(range(len(top_20_weight)), top_20_weight['feature'])
plt.xlabel('Importance Score (Weight)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Weight)')
plt.gca().invert_yaxis()

# Plot top 20 features by cover  
top_20_cover = df_cover.head(20)
plt.subplot(3, 1, 3)
plt.barh(range(len(top_20_cover)), top_20_cover['importance_cover'])
plt.yticks(range(len(top_20_cover)), top_20_cover['feature'])
plt.xlabel('Importance Score (Cover)')
plt.title('Top 20 Features - XGBoost Global Feature Importance (Cover)')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

## 1.1 Results & interpretation
We extract the global feature importance from the trained XGBoost model and display the top features based on the `gain` metric, as well as the `weight` metric. A high `gain` indicates that the feature improves the model's predictive power when used for splits: the model becomes better at distinguishing between drop-outs and non-drop-outs. A high `weight` indicates that the feature is frequently used for splits in the decision trees. For complementary information we also assess `cover`, which measures the relative coverage of samples affected by splits on a feature (i.e. a feature's influence). 

Overall (based on both gain and weight), the most important features are course result related: `Total Credits Block A`, `Average Grade A` and `Potential Credits A Missing Flag`. This suggests that students' academic performance in Block A is a key factor associated with drop-out risk. Note that the feature `Potential Credits A Missing Flag` does not have a high weight, indicating that it might only be important for a subset of students.

Examining the top feature by `cover`, we see a lot of degrees appearing. This tells us that the model segments students by degree, indicating that the model considers degree-level differences important when predicting drop-out risk: different degree programs have distinct dropout patterns. 

## 2. Permutation Importance

Permutation importance measures how much the model's performance (Average Precision/PR-AUC) decreases when each feature is randomly shuffled. This approach is more robust than tree-based feature importance because:
1. It's model-agnostic and based on actual prediction performance
2. It's less biased toward high-cardinality categorical features
3. It captures feature interactions better

In [ ]:
# Perform permutation importance analysis
# Load the test data first
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Define target and feature columns
target_col = 'sdo5_degree_drop_out'
exclude_cols = ['set', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Extract test set
test_df = df[df['set'] == 'test'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# Perform permutation importance on test set
# Using multiple repetitions for more robust estimates
perm_importance = permutation_importance(
    final_model, 
    X_test, 
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='average_precision'  # Using same metric as model tuning
)

# Create DataFrame with results
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

In [ ]:
# Visualize permutation importance
plt.figure(figsize=(12, 8))

# Plot top 20 features
top_20_perm = perm_df.head(20)
plt.barh(range(len(top_20_perm)), top_20_perm['importance_mean'], 
         xerr=top_20_perm['importance_std'])
plt.yticks(range(len(top_20_perm)), top_20_perm['feature'])
plt.xlabel('Decrease in Average Precision (PR-AUC) Score')
plt.title('Top 20 Features - Permutation Importance on Test Set')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2.1 Results & interpretation

All features related to course results in Block A appear in the top 10 most important features, confirming the findings from the XGBoost feature importance. Notably, `Total Credits Block A` and `Average Grade A` are again among the most important features, indicating that students' academic performance in Block A is a key factor associated with drop-out risk.

Apart from course results, we also the `has_dsa` feature (indicating whether the degree the student is enrolled in is enforcing a binding study advice) and the `gender` feature. These two features did not appear as prominently during the XGBoost importance analysis, indicating that they might be underutilised but important. The importance of `has_dsa` suggests that students in degrees with a binding study advice might have different drop-out patterns, potentially due to increased pressure or support. The importance of `gender` indicates that there may be gender differences in drop-out risk.

## 3. SHAP Global Summary

SHAP (SHapley Additive exPlanations) values provide a unified framework for interpreting model predictions. We'll create a global summary showing the average impact of each feature.

In [ ]:
# SHAP Analysis
# Create SHAP explainer for XGBoost
explainer = shap.TreeExplainer(final_model)

# Calculate SHAP values for test set (sample if too large for memory)
# Using a sample for computational efficiency
sample_size = min(5000, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

print(f"Calculating SHAP values for {sample_size} test samples...")
shap_values = explainer.shap_values(X_test_sample)
print("SHAP values calculated successfully!")

# Global feature importance based on mean absolute SHAP values
shap_importance = pd.DataFrame({
    'feature': X_test_sample.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("Top 20 features by SHAP importance:")
print(shap_importance.head(20))

In [ ]:
# Standalone SHAP Summary Plot for detailed analysis
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title('SHAP Summary Plot - Feature Impact Distribution\n(Red = High feature values, Blue = Low feature values)', 
          fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 3.1 SHAP Results & Interpretation

SHAP values provide the most detailed and theoretically sound explanation of our model's predictions. Each SHAP value represents the contribution of a specific feature to moving the prediction away from the expected baseline.

Key insights from the SHAP analysis:

**Most Important Features (by mean absolute SHAP value):**
- Similar to previous methods, Block A academic performance features dominate
- The SHAP summary plot shows both the importance and the directional impact of features
- Positive SHAP values (red) push predictions toward dropout, negative values (blue) push toward retention

**Feature Impact Patterns:**
- **Total Credits Block A**: Lower credits strongly associated with higher dropout risk
- **Average Grade A**: Lower grades associated with higher dropout risk  
- **Course result flags**: Missing course results are significant risk factors

**SHAP Advantages:**
1. **Directional insights**: Unlike other methods, SHAP shows whether high/low feature values increase or decrease dropout risk
2. **Individual-level explanations**: Each prediction can be fully explained 
3. **Interaction effects**: SHAP captures how features work together
4. **Theoretical soundness**: Based on cooperative game theory with desirable mathematical properties

## 4. Comparative Analysis

Let's compare the results from all three methods to identify the most robust dropout risk factors.

In [ ]:
# Comparative analysis of all three methods
# Combine results from all three approaches
comparison_df = pd.DataFrame({
    'feature': feature_cols
})

# Add XGBoost importance (gain)
comparison_df = comparison_df.merge(
    df_gain.rename(columns={'importance_gain': 'xgb_gain'})[['feature', 'xgb_gain']], 
    on='feature', how='left'
).fillna(0)

# Add XGBoost importance (weight)  
comparison_df = comparison_df.merge(
    df_weight.rename(columns={'importance_weight': 'xgb_weight'})[['feature', 'xgb_weight']], 
    on='feature', how='left'
).fillna(0)

# Add permutation importance
comparison_df = comparison_df.merge(
    perm_df.rename(columns={'importance_mean': 'perm_importance'})[['feature', 'perm_importance']], 
    on='feature', how='left'
).fillna(0)

# Add SHAP importance
comparison_df = comparison_df.merge(
    shap_importance.rename(columns={'shap_importance': 'shap_importance'})[['feature', 'shap_importance']], 
    on='feature', how='left'
).fillna(0)

# Normalize all importance scores to 0-1 scale for comparison
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

importance_cols = ['xgb_gain', 'xgb_weight', 'perm_importance', 'shap_importance']
comparison_df[importance_cols] = scaler.fit_transform(comparison_df[importance_cols])

# Calculate average importance across all methods
comparison_df['avg_importance'] = comparison_df[importance_cols].mean(axis=1)
comparison_df = comparison_df.sort_values('avg_importance', ascending=False)

# Display top 20 features across all methods
print("Top 20 Features - Comparative Analysis (Normalized 0-1 Scale):")
print(comparison_df.head(20)[['feature'] + importance_cols + ['avg_importance']].round(3))

In [ ]:
# Visualize comparative analysis
plt.figure(figsize=(15, 10))

# Heatmap of top 20 features across all methods
top_20_comparison = comparison_df.head(20)

plt.subplot(2, 1, 1)
heatmap_data = top_20_comparison[importance_cols].values.T
plt.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Normalized Importance Score')
plt.yticks(range(len(importance_cols)), 
           ['XGBoost (Gain)', 'XGBoost (Weight)', 'Permutation', 'SHAP'])
plt.xticks(range(len(top_20_comparison)), 
           top_20_comparison['feature'], rotation=45, ha='right')
plt.title('Top 20 Features - Importance Heatmap Across Methods')

# Bar plot of average importance
plt.subplot(2, 1, 2)
plt.barh(range(len(top_20_comparison)), top_20_comparison['avg_importance'])
plt.yticks(range(len(top_20_comparison)), top_20_comparison['feature'])
plt.xlabel('Average Normalized Importance Score')
plt.title('Top 20 Features - Average Importance Across All Methods')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Summary of most consistent important features
print("\nMost Consistent Important Features (appearing in top 10 of multiple methods):")
top_10_per_method = {
    'XGBoost_Gain': set(df_gain.head(10)['feature']),
    'XGBoost_Weight': set(df_weight.head(10)['feature']), 
    'Permutation': set(perm_df.head(10)['feature']),
    'SHAP': set(shap_importance.head(10)['feature'])
}

# Find features appearing in top 10 of at least 3 methods
all_features = set()
for method_features in top_10_per_method.values():
    all_features.update(method_features)

consistent_features = []
for feature in all_features:
    count = sum(1 for method_features in top_10_per_method.values() 
                if feature in method_features)
    if count >= 3:
        consistent_features.append((feature, count))

consistent_features.sort(key=lambda x: x[1], reverse=True)

for feature, count in consistent_features:
    print(f"- {feature}: appears in top 10 of {count}/4 methods")